In [ ]:
from __future__ import annotations

from pathlib import Path
import csv
import hashlib
import json
from datetime import datetime, timezone
import logging

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

# --- PyArrow/Pandas extension-type double-registration guard (Jupyter-safe) ---
try:
    if not getattr(pa, "_pandas_ext_registered", False):
        import pandas.core.arrays.arrow.extension_types  # noqa: F401

        pa._pandas_ext_registered = True
except Exception:
    # Non-fatal; parquet may still work after kernel restart.
    pass


# ============================================================
# CONFIG
# ============================================================
RAW_CSV_PATH = Path("data/raw/complaints.csv")
OUT_DIR = Path("data/clean")

PIPELINE_VERSION = "v2.4.1-postgres-ready"

LOW_MEMORY = False
ENABLE_INPUT_HASH = False

# Core outputs
EXPORT_PARQUET_SOR = True
EXPORT_POSTGRES_CSV = True
EXPORT_CSV_SAMPLE = True
CSV_SAMPLE_N = 100_000
CSV_SAMPLE_FILENAME = "sample_scoped_100k.csv"

EXPORT_DQ_UNMAPPED_PRODUCTS_TOP = True

# PostgreSQL-oriented CSV export (primary DB import artifacts)
POSTGRES_STAGE_FILENAME = "stg_complaints_scoped_postgres.csv"
POSTGRES_TAG_FILENAME = "bridge_complaint_tag_scoped_postgres.csv"
POSTGRES_CSV_ENCODING = "utf-8"
POSTGRES_CSV_LINETERMINATOR = "\n"
POSTGRES_CSV_NA_REP = ""

# Legacy SQL Server-style bulk export retained as optional compatibility path
# (disabled by default and not primary artifact)
EXPORT_SQLSERVER_BULK_COMPAT = False
SQLSERVER_BULK_DELIM = "\x1f"
SQLSERVER_BULK_FILENAME = "stg_complaints_scoped_bulk.txt"
SQLSERVER_BULK_ENCODING = "utf-8"
SQLSERVER_BULK_LINETERMINATOR = "\n"
SQLSERVER_BULK_QUOTING = csv.QUOTE_NONE
SQLSERVER_BULK_ESCAPECHAR = "\\"
SQLSERVER_BULK_NA_REP = ""

YEARS_BACK = 3
ALLOWED_DOMAINS = {"Credit Reporting", "Debt Collection", "Checking / Savings", "Mortgage"}
SCOPE_REQUIRE_NARRATIVE = False

ASOF_MODE = "TODAY"  # "TODAY" or "MAX_DATE_RECEIVED"
FUTURE_DATE_TOLERANCE_DAYS = 2
DATE_PARSE_MODE = "LOCKED"
DATE_FORMAT = "%m/%d/%Y"
DATE_LOCK_FAIL_RATE_THRESHOLD = 0.05

PRODUCT_DOMAIN_MAP = {
    "credit reporting or other personal consumer reports": "Credit Reporting",
    "credit reporting, credit repair services, or other personal consumer reports": "Credit Reporting",
    "debt collection": "Debt Collection",
    "checking or savings account": "Checking / Savings",
    "mortgage": "Mortgage",
}
DEFAULT_DOMAIN = "Other"

FOCUS_CANDIDATE_SUBSTRINGS = [
    "credit reporting",
    "consumer report",
    "debt collection",
    "checking",
    "savings",
    "mortgage",
]
UNMAPPED_FOCUS_FAIL_RATE = 0.10
FAIL_ON_UNMAPPED_FOCUS = False

CHANNEL_CANON = {
    "web": "web",
    "phone": "phone",
    "postal mail": "postal mail",
    "referral": "referral",
    "email": "email",
}
CHANNEL_OTHER = "other"
HEAVY_CHANNELS = {"phone", "postal mail", "referral"}

USECOLS = [
    "Date received",
    "Date sent to company",
    "Product",
    "Issue",
    "Company",
    "State",
    "ZIP code",
    "Submitted via",
    "Company response to consumer",
    "Timely response?",
    "Consumer disputed?",
    "Complaint ID",
    "Sub-product",
    "Sub-issue",
    "Consumer consent provided?",
    "Company public response",
    "Consumer complaint narrative",
    "Tags",
]


# LOGGING


In [2]:

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)
logger = logging.getLogger(__name__)

In [3]:
# ============================================================
# HELPERS
# ============================================================
def utc_now_iso() -> str:
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat()


def run_id_utc() -> str:
    return datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")


def file_sha256(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def write_json(path: Path, obj: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


def safe_col(df: pd.DataFrame, name: str) -> pd.Series:
    if name in df.columns:
        return df[name]
    return pd.Series(pd.NA, index=df.index, dtype="string")


def yes_no_to_flag(s: pd.Series) -> pd.Series:
    s2 = s.astype("string").str.strip().str.lower()
    out = s2.map({"yes": 1, "no": 0})
    return out.astype("Int64")


def canon_channel_series(s: pd.Series) -> pd.Series:
    x = s.astype("string").str.strip().str.lower()
    out = x.map(CHANNEL_CANON).fillna(CHANNEL_OTHER)
    out = out.mask(out.eq(""), CHANNEL_OTHER)
    return out.astype("string")


def _read_header_cols(path: Path) -> list[str]:
    with path.open("r", encoding="utf-8", errors="replace", newline="") as f:
        reader = csv.reader(f)
        return next(reader)


def _read_csv_fast(path: Path) -> tuple[pd.DataFrame, dict]:
    wanted = [c.strip().replace("\ufeff", "") for c in USECOLS]
    header_cols = [c.strip().replace("\ufeff", "") for c in _read_header_cols(path)]
    header_set = set(header_cols)
    usecols_list = [c for c in wanted if c in header_set]
    missing_usecols = sorted(set(wanted) - set(usecols_list))

    # C engine + bad-line skipping for malformed-row tolerance.
    df = pd.read_csv(
        path,
        usecols=usecols_list,
        engine="c",
        low_memory=LOW_MEMORY,
        on_bad_lines="skip",
    )
    df.columns = [c.strip().replace("\ufeff", "") for c in df.columns]

    meta = {
        "requested_usecols_count": len(wanted),
        "resolved_usecols_count": len(usecols_list),
        "missing_usecols_from_input_header": missing_usecols,
        "bad_line_handling": "on_bad_lines=skip",
    }
    return df, meta


def _parse_dates(df: pd.DataFrame) -> tuple[pd.Series, pd.Series, dict]:
    parse_fallback_used = False
    dr_fail_locked = None
    ds_fail_locked = None
    fallback_triggered_date_received = False
    fallback_triggered_date_sent_to_company = False

    if DATE_PARSE_MODE.upper() == "LOCKED":
        dr_locked = pd.to_datetime(df["Date received"], format=DATE_FORMAT, errors="coerce")
        ds_locked = pd.to_datetime(df["Date sent to company"], format=DATE_FORMAT, errors="coerce")

        dr_fail_locked = float(dr_locked.isna().mean())
        ds_fail_locked = float(ds_locked.isna().mean())

        dr = dr_locked
        ds = ds_locked

        if dr_fail_locked > DATE_LOCK_FAIL_RATE_THRESHOLD:
            dr = pd.to_datetime(df["Date received"], errors="coerce")
            parse_fallback_used = True
            fallback_triggered_date_received = True
        if ds_fail_locked > DATE_LOCK_FAIL_RATE_THRESHOLD:
            ds = pd.to_datetime(df["Date sent to company"], errors="coerce")
            parse_fallback_used = True
            fallback_triggered_date_sent_to_company = True
    else:
        dr = pd.to_datetime(df["Date received"], errors="coerce")
        ds = pd.to_datetime(df["Date sent to company"], errors="coerce")

    meta = {
        "mode": DATE_PARSE_MODE,
        "format_locked": DATE_FORMAT if DATE_PARSE_MODE.upper() == "LOCKED" else None,
        "lock_fail_rate_threshold": DATE_LOCK_FAIL_RATE_THRESHOLD,
        "parse_fallback_used": parse_fallback_used,
        "fallback_triggered_date_received": fallback_triggered_date_received,
        "fallback_triggered_date_sent_to_company": fallback_triggered_date_sent_to_company,
        "date_received_fail_rate_locked": dr_fail_locked,
        "date_sent_to_company_fail_rate_locked": ds_fail_locked,
        "date_received_fail_rate_final": float(dr.isna().mean()),
        "date_sent_to_company_fail_rate_final": float(ds.isna().mean()),
    }
    return dr, ds, meta


def _bool_to_int64(x: pd.Series) -> pd.Series:
    return x.astype("boolean").astype("Int64")


def _sanitize_text_for_csv(s: pd.Series) -> pd.Series:
    ss = s.astype("string")
    ss = ss.str.replace("\r\n", " ", regex=False).str.replace("\n", " ", regex=False).str.replace("\r", " ", regex=False)
    return ss


def _sanitize_textlike_cols_for_csv(df: pd.DataFrame) -> None:
    string_cols = list(df.select_dtypes(include=["string", "object"]).columns)
    for c in string_cols:
        df[c] = _sanitize_text_for_csv(df[c])


def _assert_no_object_cols_in_stage(df: pd.DataFrame, stage_name: str) -> None:
    obj_cols = list(df.select_dtypes(include=["object"]).columns)
    if obj_cols:
        raise TypeError(
            f"{stage_name} has object dtype columns (should be StringDtype or typed numeric/date). "
            f"Fix stage builder to cast appropriately for: {obj_cols}"
        )


def _asof_start(df: pd.DataFrame) -> tuple[pd.Timestamp, pd.Timestamp, dict]:
    today_utc_date = datetime.now(timezone.utc).date()
    today_floor = pd.Timestamp(today_utc_date)
    asof_today = today_floor + pd.Timedelta(days=1)

    max_dr = df["date_received"].max()
    if pd.isna(max_dr):
        raise ValueError("date_received is entirely NaT after parsing; check DATE_FORMAT / DATE_PARSE_MODE.")

    max_allowed = asof_today + pd.Timedelta(days=FUTURE_DATE_TOLERANCE_DAYS)
    if max_dr >= max_allowed:
        raise ValueError(f"date_received has future dates: max={max_dr.date()} >= allowed={max_allowed.date()}.")

    if ASOF_MODE.upper() == "TODAY":
        asof = asof_today
        anchor_used = "TODAY_UTC"
    else:
        asof = pd.Timestamp(max_dr.date()) + pd.Timedelta(days=1)
        anchor_used = "MAX_DATE_RECEIVED"

    start = asof - pd.DateOffset(years=YEARS_BACK)
    meta = {
        "asof_mode": ASOF_MODE,
        "anchor_used": anchor_used,
        "today_utc": str(today_utc_date),
        "max_date_received": str(pd.Timestamp(max_dr).date()),
        "future_date_tolerance_days": FUTURE_DATE_TOLERANCE_DAYS,
        "asof_date": str(asof.date()),
        "start_date": str(start.date()),
    }
    return asof, start, meta


def _is_focus_candidate(product_std: pd.Series) -> pd.Series:
    s = product_std.astype("string").fillna("").str.lower()
    mask = pd.Series(False, index=s.index)
    for sub in FOCUS_CANDIDATE_SUBSTRINGS:
        mask = mask | s.str.contains(sub, regex=False)
    return mask


def _dedupe_best_record(df: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    df = df.copy()

    has_id = df["complaint_id"].notna() & df["complaint_id"].ne("")
    df_with_id = df.loc[has_id].copy()
    df_no_id = df.loc[~has_id].copy()

    dup_mask = df_with_id["complaint_id"].duplicated(keep=False)
    df_dups = df_with_id.loc[dup_mask].copy()
    df_uniqs = df_with_id.loc[~dup_mask].copy()

    if len(df_dups):
        key_cols = [
            "date_received",
            "date_sent_to_company",
            "Company",
            "Product",
            "Issue",
            "State",
            "ZIP code",
            "Submitted via",
            "Company response to consumer",
            "Timely response?",
            "Consumer disputed?",
        ]
        df_dups["_present_key_fields"] = df_dups[key_cols].notna().sum(axis=1).astype("int16")
        df_dups["_dr2"] = df_dups["date_received"].fillna(pd.Timestamp("1900-01-01"))
        df_dups["_ds2"] = df_dups["date_sent_to_company"].fillna(pd.Timestamp("1900-01-01"))
        df_dups["_row_idx"] = df_dups.index.astype("int64")

        best = (
            df_dups.sort_values(
                ["complaint_id", "_dr2", "_ds2", "_present_key_fields", "_row_idx"],
                ascending=[True, False, False, False, False],
                kind="mergesort",
            )
            .drop_duplicates(subset=["complaint_id"], keep="first")
            .drop(columns=["_present_key_fields", "_dr2", "_ds2", "_row_idx"], errors="ignore")
        )

        df_with_id_out = pd.concat([df_uniqs, best], axis=0)
    else:
        df_with_id_out = df_with_id

    out = pd.concat([df_with_id_out, df_no_id], axis=0).reset_index(drop=True)

    dup_remaining = int(
        out.loc[out["complaint_id"].notna() & out["complaint_id"].ne(""), "complaint_id"]
        .duplicated()
        .sum()
    )

    meta = {
        "dedupe_method": "best_record_sort_tiebreak_only_dups",
        "dedupe_tiebreakers": [
            "date_received desc",
            "date_sent_to_company desc",
            "present_key_fields desc",
            "row_index desc",
        ],
        "rows_with_id": int(len(df_with_id)),
        "rows_with_id_duplicated_subset": int(len(df_dups)),
        "unique_ids_with_dupes": int(df_dups["complaint_id"].nunique()) if len(df_dups) else 0,
        "duplicate_ids_remaining_post_dedup": dup_remaining,
    }
    return out, meta


def _build_stage_scoped(
    dfx: pd.DataFrame, asof: pd.Timestamp, date_as_string: bool
) -> pd.DataFrame:
    if date_as_string:
        # Keep null dates as <NA> (not "nan")
        date_received = dfx["date_received"].dt.strftime("%Y-%m-%d").astype("string")
        date_sent = dfx["date_sent_to_company"].dt.strftime("%Y-%m-%d").astype("string")
        as_of_date = pd.Series(asof.strftime("%Y-%m-%d"), index=dfx.index, dtype="string")
    else:
        date_received = dfx["date_received"].dt.normalize()
        date_sent = dfx["date_sent_to_company"].dt.normalize()
        as_of_date = pd.Series(asof.normalize(), index=dfx.index)

    return pd.DataFrame(
        {
            "complaint_id": dfx["complaint_id"].astype("string"),
            "date_received": date_received,
            "date_sent_to_company": date_sent,
            "as_of_date": as_of_date,
            "company_raw": dfx["Company"].astype("string"),
            "company_std": dfx["company_std"].astype("string"),
            "product_raw": dfx["Product"].astype("string"),
            "product_std": dfx["product_std"].astype("string"),
            "product_domain": dfx["product_domain"].astype("string"),
            "domain_is_focus_flag": dfx["domain_is_focus_flag"].astype("Int64"),
            "issue_raw": dfx["Issue"].astype("string"),
            "issue_std": dfx["issue_std"].astype("string"),
            "sub_product_std": dfx["sub_product_std"].astype("string"),
            "sub_issue_std": dfx["sub_issue_std"].astype("string"),
            "state": dfx["State"].astype("string"),
            "zip3": dfx["zip3"].astype("string"),
            "zip5": dfx["zip5"].astype("string"),
            "channel_std": dfx["channel_std"].astype("string"),
            "company_response_to_consumer": dfx["Company response to consumer"].astype("string"),
            "company_public_response": dfx["company_public_response"].astype("string"),
            "timely_flag": dfx["timely_flag"].astype("Int64"),
            "disputed_flag": dfx["disputed_flag"].astype("Int64"),
            "has_narrative_flag": dfx["has_narrative_flag"].astype("Int64"),
            "narrative_length_chars": dfx["narrative_length_chars"].astype("Int64"),
            "intake_to_routing_days": dfx["intake_to_routing_days"].astype("Int64"),
            "lag_negative_flag": _bool_to_int64(dfx["lag_negative_flag"]),
            "days_since_received": dfx["days_since_received"].astype("Int64"),
            "days_since_received_bucket": dfx["days_since_received_bucket"].astype("string"),
            "heavy_channel_flag": dfx["heavy_channel_flag"].astype("Int64"),
            "consent_std": dfx["consent_std"].astype("string"),
            "long_narrative_flag": dfx["long_narrative_flag"].astype("Int64"),
            "untimely_flag": dfx["untimely_flag"].astype("Int64"),
            "dispute_flag": dfx["dispute_flag"].astype("Int64"),
            "effort_score": dfx["effort_score"].astype("Int64"),
            "effort_unknown_flag": dfx["effort_unknown_flag"].astype("Int64"),
        }
    )


# ============================================================
# MAIN
# ============================================================
def main() -> None:
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    run_id = run_id_utc()
    input_hash = file_sha256(RAW_CSV_PATH) if ENABLE_INPUT_HASH else None

    logger.info("Reading input CSV...")
    df, read_meta = _read_csv_fast(RAW_CSV_PATH)
    rows_extracted_input = int(len(df))
    logger.info("Rows extracted: %s", rows_extracted_input)

    required = [
        "Date received",
        "Product",
        "Issue",
        "Company",
        "State",
        "ZIP code",
        "Submitted via",
        "Date sent to company",
        "Company response to consumer",
        "Timely response?",
        "Consumer disputed?",
        "Complaint ID",
    ]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    dr, ds, date_parsing_meta = _parse_dates(df)
    df["date_received"] = dr
    df["date_sent_to_company"] = ds
    df["complaint_id"] = df["Complaint ID"].astype("string").str.strip()

    valid_id_mask = df["complaint_id"].notna() & df["complaint_id"].ne("")
    blank_id_count = int((~valid_id_mask).sum())
    duplicates_raw_count = int(df.loc[valid_id_mask, "complaint_id"].duplicated().sum())

    logger.info("Running dedupe...")
    df, dedupe_meta = _dedupe_best_record(df)
    rows_after_dedup = int(len(df))

    has_id_post = df["complaint_id"].notna() & df["complaint_id"].ne("")
    post_dedup_rows_with_id = int(df.loc[has_id_post].shape[0])
    post_dedup_unique_ids = int(df.loc[has_id_post, "complaint_id"].nunique(dropna=True))
    complaint_id_unique_ok = bool(post_dedup_unique_ids == post_dedup_rows_with_id)

    asof, start, window_meta = _asof_start(df)
    in_window = df["date_received"].between(start, asof, inclusive="left").fillna(False)
    removed_not_in_window = int((~in_window).sum())
    df = df.loc[in_window].copy()
    rows_in_window_3y = int(len(df))

    df["company_std"] = df["Company"].astype("string").str.strip().str.upper()
    df["issue_std"] = df["Issue"].astype("string").str.strip()
    df["product_std"] = df["Product"].astype("string").str.strip().str.lower()
    df["channel_std"] = canon_channel_series(df["Submitted via"])

    df["sub_product_std"] = safe_col(df, "Sub-product").astype("string").str.strip()
    df["sub_issue_std"] = safe_col(df, "Sub-issue").astype("string").str.strip()
    df["consent_std"] = safe_col(df, "Consumer consent provided?").astype("string").str.strip().str.lower()
    df["company_public_response"] = safe_col(df, "Company public response").astype("string").str.strip()

    narr = safe_col(df, "Consumer complaint narrative").astype("string")
    narr_len = narr.str.len()
    df["has_narrative_flag"] = (narr.notna() & narr_len.fillna(0).gt(0)).astype("Int64")
    df["narrative_length_chars"] = narr_len.astype("Int64")
    del narr, narr_len  # memory cleanup for large inputs

    df["timely_flag"] = yes_no_to_flag(df["Timely response?"])
    df["disputed_flag"] = yes_no_to_flag(df["Consumer disputed?"])

    df["product_domain"] = df["product_std"].map(PRODUCT_DOMAIN_MAP)
    unmapped_mask = df["product_domain"].isna()
    focus_candidate = _is_focus_candidate(df["product_std"])

    focus_candidate_count = int(focus_candidate.sum())
    unmapped_focus_count = int((focus_candidate & unmapped_mask).sum())
    unmapped_focus_rate = float(unmapped_focus_count / focus_candidate_count) if focus_candidate_count else 0.0

    if EXPORT_DQ_UNMAPPED_PRODUCTS_TOP:
        (
            df.loc[focus_candidate & unmapped_mask, "product_std"]
            .value_counts()
            .head(200)
            .rename_axis("product_std")
            .reset_index(name="count")
            .to_csv(OUT_DIR / "dq_unmapped_focus_products_top200.csv", index=False)
        )

    df["product_domain"] = df["product_domain"].fillna(DEFAULT_DOMAIN).astype("string")
    if FAIL_ON_UNMAPPED_FOCUS and (unmapped_focus_rate >= UNMAPPED_FOCUS_FAIL_RATE):
        raise ValueError(f"Unmapped focus-candidate product rate too high: {unmapped_focus_rate:.2%}.")

    lag_days = (df["date_sent_to_company"] - df["date_received"]).dt.days
    df["lag_negative_flag"] = lag_days.lt(0)
    df["intake_to_routing_days"] = lag_days.where(lag_days.ge(0)).astype("Int64")

    missing_lag_input_rate = float((df["date_received"].isna() | df["date_sent_to_company"].isna()).mean())
    missing_complaint_id_count = int((df["complaint_id"].isna() | df["complaint_id"].eq("")).sum())
    negative_lag_count_in_window = int(df["lag_negative_flag"].fillna(False).sum())

    in_dom = df["product_domain"].isin(ALLOWED_DOMAINS).fillna(False)
    scope_mask = in_dom
    if SCOPE_REQUIRE_NARRATIVE:
        scope_mask = scope_mask & df["has_narrative_flag"].eq(1)

    missing_narrative_rate_in_window = float((df["has_narrative_flag"] != 1).mean()) if len(df) else 0.0
    out_of_scope_domain_rate_in_window = float((~in_dom).mean()) if len(df) else 0.0
    in_scope_rate_in_window = float(scope_mask.mean()) if len(df) else 0.0

    removed_out_of_scope_domain_in_window = int((~in_dom).sum())
    diagnostic_missing_narrative_in_window = int((df["has_narrative_flag"] != 1).sum())
    scope_removed_missing_narrative = diagnostic_missing_narrative_in_window if SCOPE_REQUIRE_NARRATIVE else 0

    df_scoped = df.loc[scope_mask].copy()
    rows_scoped = int(len(df_scoped))

    scope_diagnostics_rates = {
        "out_of_window_rate": float(removed_not_in_window / rows_after_dedup) if rows_after_dedup else 0.0,
        "missing_narrative_rate_in_window": missing_narrative_rate_in_window,
        "out_of_scope_domain_rate_in_window": out_of_scope_domain_rate_in_window,
        "in_scope_rate_in_window": in_scope_rate_in_window,
    }
    narrative_present_rate_scoped = float(df_scoped["has_narrative_flag"].fillna(0).mean()) if rows_scoped else 0.0

    df_scoped["as_of_date"] = asof
    df_scoped["days_since_received"] = (df_scoped["as_of_date"] - df_scoped["date_received"]).dt.days.astype("Int64")
    df_scoped["days_since_received_bucket"] = pd.cut(
        df_scoped["days_since_received"].astype("float"),
        bins=[-1, 7, 15, 30, 60, 10_000],
        labels=["0–7", "8–15", "16–30", "31–60", "60+"],
    )

    zip_raw = df_scoped["ZIP code"].astype("string")
    df_scoped["zip5"] = zip_raw.str.extract(r"(\d{5})", expand=False).astype("string")
    df_scoped["zip3"] = df_scoped["zip5"].str.slice(0, 3)
    df_scoped["heavy_channel_flag"] = df_scoped["channel_std"].isin(HEAVY_CHANNELS).astype("Int64")
    df_scoped["domain_is_focus_flag"] = df_scoped["product_domain"].isin(ALLOWED_DOMAINS).astype("Int64")

    narr_len_scoped = df_scoped.loc[df_scoped["has_narrative_flag"].eq(1), "narrative_length_chars"].dropna()
    narrative_p75_chars = int(narr_len_scoped.astype("int").quantile(0.75)) if len(narr_len_scoped) else 0
    write_json(OUT_DIR / "effort_thresholds.json", {"narrative_p75_chars": narrative_p75_chars})

    df_scoped["long_narrative_flag"] = pd.Series(pd.NA, index=df_scoped.index, dtype="Int64")
    if narrative_p75_chars > 0 and rows_scoped > 0:
        m = df_scoped["has_narrative_flag"].eq(1) & df_scoped["narrative_length_chars"].notna()
        df_scoped.loc[m, "long_narrative_flag"] = (
            df_scoped.loc[m, "narrative_length_chars"].astype("int") >= narrative_p75_chars
        ).astype("Int64")

    df_scoped["untimely_flag"] = df_scoped["timely_flag"].map({0: 1, 1: 0}).astype("Int64")
    df_scoped["dispute_flag"] = df_scoped["disputed_flag"].map({1: 1, 0: 0}).astype("Int64")
    df_scoped["effort_unknown_flag"] = (
        df_scoped["untimely_flag"].isna()
        | df_scoped["dispute_flag"].isna()
        | df_scoped["long_narrative_flag"].isna()
        | df_scoped["heavy_channel_flag"].isna()
    ).astype("Int64")
    df_scoped["effort_score"] = (
        df_scoped["untimely_flag"].fillna(0).astype("int")
        + df_scoped["dispute_flag"].fillna(0).astype("int")
        + df_scoped["long_narrative_flag"].fillna(0).astype("int")
        + df_scoped["heavy_channel_flag"].fillna(0).astype("int")
    ).astype("Int64")

    tags_col = safe_col(df_scoped, "Tags").astype("string")
    tmp = df_scoped.assign(Tags=tags_col)[["complaint_id", "Tags"]]
    tmp = tmp.dropna(subset=["complaint_id"])
    tmp = tmp[tmp["complaint_id"].astype("string").str.strip().ne("")]
    tmp = tmp.dropna(subset=["Tags"])
    tmp = tmp[tmp["Tags"].astype("string").str.strip().ne("")]
    bridge_scoped = tmp.assign(tag=tmp["Tags"].str.split(",")).explode("tag")
    bridge_scoped["tag"] = bridge_scoped["tag"].astype("string").str.strip()
    bridge_scoped = bridge_scoped[bridge_scoped["tag"].notna() & (bridge_scoped["tag"] != "")]
    bridge_scoped = bridge_scoped[["complaint_id", "tag"]].copy()

    # Explicit dtype normalization for bridge before any export
    bridge_scoped["complaint_id"] = bridge_scoped["complaint_id"].astype("string")
    bridge_scoped["tag"] = bridge_scoped["tag"].astype("string")

    # Build export stages
    stg_scoped_parquet = _build_stage_scoped(df_scoped, asof=asof, date_as_string=False)
    stg_scoped_postgres = _build_stage_scoped(df_scoped, asof=asof, date_as_string=True)

    # CSV safety sanitization for text columns
    _sanitize_textlike_cols_for_csv(stg_scoped_postgres)
    _sanitize_textlike_cols_for_csv(bridge_scoped)

    # Guardrails
    _assert_no_object_cols_in_stage(stg_scoped_postgres, "PostgreSQL stage")
    _assert_no_object_cols_in_stage(bridge_scoped, "PostgreSQL tag bridge")
    _assert_no_object_cols_in_stage(stg_scoped_parquet, "Parquet stage")

    negative_lag_count_scoped = int(df_scoped["lag_negative_flag"].fillna(False).sum())
    negative_lag_rate_scoped = float(df_scoped["lag_negative_flag"].fillna(False).mean()) if rows_scoped else 0.0

    out_paths: dict[str, str] = {}

    # Parquet SOR
    p1 = OUT_DIR / "stg_complaints_scoped.parquet"
    p2 = OUT_DIR / "bridge_complaint_tag_scoped.parquet"
    if EXPORT_PARQUET_SOR:
        try:
            table1 = pa.Table.from_pandas(stg_scoped_parquet, preserve_index=False)
            pq.write_table(table1, p1)
            table2 = pa.Table.from_pandas(bridge_scoped, preserve_index=False)
            pq.write_table(table2, p2)
            out_paths["stg_scoped_parquet"] = str(p1)
            out_paths["bridge_scoped_parquet"] = str(p2)
        except Exception as e:
            raise RuntimeError("Parquet export failed using pyarrow direct write.") from e

    # Primary PostgreSQL CSV artifacts
    if EXPORT_POSTGRES_CSV:
        pg_stage_path = OUT_DIR / POSTGRES_STAGE_FILENAME
        pg_tag_path = OUT_DIR / POSTGRES_TAG_FILENAME

        stg_scoped_postgres.to_csv(
            pg_stage_path,
            index=False,
            encoding=POSTGRES_CSV_ENCODING,
            lineterminator=POSTGRES_CSV_LINETERMINATOR,
            na_rep=POSTGRES_CSV_NA_REP,
        )
        bridge_scoped.to_csv(
            pg_tag_path,
            index=False,
            encoding=POSTGRES_CSV_ENCODING,
            lineterminator=POSTGRES_CSV_LINETERMINATOR,
            na_rep=POSTGRES_CSV_NA_REP,
        )

        out_paths["stg_scoped_postgres_csv"] = str(pg_stage_path)
        out_paths["bridge_scoped_postgres_csv"] = str(pg_tag_path)

    # Optional legacy SQL Server-compatible bulk export
    if EXPORT_SQLSERVER_BULK_COMPAT:
        bulk_path = OUT_DIR / SQLSERVER_BULK_FILENAME
        stg_sqlserver_bulk = stg_scoped_postgres.copy()
        for c in stg_sqlserver_bulk.select_dtypes(include=["string"]).columns:
            stg_sqlserver_bulk[c] = stg_sqlserver_bulk[c].str.replace(SQLSERVER_BULK_DELIM, " ", regex=False)

        stg_sqlserver_bulk.to_csv(
            bulk_path,
            index=False,
            sep=SQLSERVER_BULK_DELIM,
            encoding=SQLSERVER_BULK_ENCODING,
            lineterminator=SQLSERVER_BULK_LINETERMINATOR,
            quoting=SQLSERVER_BULK_QUOTING,
            escapechar=SQLSERVER_BULK_ESCAPECHAR,
            na_rep=SQLSERVER_BULK_NA_REP,
        )
        out_paths["sqlserver_bulk_compat_file"] = str(bulk_path)

    if EXPORT_CSV_SAMPLE:
        sample_path = OUT_DIR / CSV_SAMPLE_FILENAME
        sample = (
            stg_scoped_postgres.sample(min(CSV_SAMPLE_N, len(stg_scoped_postgres)), random_state=42)
            if len(stg_scoped_postgres)
            else stg_scoped_postgres
        )
        sample.to_csv(sample_path, index=False, encoding="utf-8")
        out_paths["sample_scoped_csv"] = str(sample_path)

    vc_channels = stg_scoped_postgres["channel_std"].dropna().value_counts().head(5)
    vc_products = stg_scoped_postgres["product_std"].dropna().value_counts().head(5)
    vc_domains = stg_scoped_postgres["product_domain"].dropna().value_counts().head(5)

    dq_run_kpis = pd.DataFrame(
        [
            {
                "run_id": run_id,
                "pipeline_version": PIPELINE_VERSION,
                "rows_raw": rows_extracted_input,
                "rows_after_dedup": rows_after_dedup,
                "rows_in_window_3y": rows_in_window_3y,
                "rows_scoped_focus": rows_scoped,
                "duplicates_raw_count_valid_ids": duplicates_raw_count,
                "blank_id_count": blank_id_count,
                "missing_complaint_id_count": missing_complaint_id_count,
                "negative_lag_count_in_window": negative_lag_count_in_window,
                "negative_lag_count_scoped": negative_lag_count_scoped,
                "negative_lag_rate_scoped": negative_lag_rate_scoped,
                "missing_lag_input_rate": missing_lag_input_rate,
                "narrative_present_rate_scoped": narrative_present_rate_scoped,
                "scope_require_narrative": int(SCOPE_REQUIRE_NARRATIVE),
                "unmapped_focus_candidate_rate_in_window": unmapped_focus_rate,
                "unmapped_focus_candidate_count": unmapped_focus_count,
                "focus_candidate_count": focus_candidate_count,
                "top5_channels_scoped": "|".join(vc_channels.index.astype("string").tolist()),
                "top5_products_scoped": "|".join(vc_products.index.astype("string").tolist()),
                "top5_product_domains_scoped": "|".join(vc_domains.index.astype("string").tolist()),
                "asof_date": str(asof.date()),
                "start_date_3y": str(start.date()),
                "asof_mode": ASOF_MODE,
            }
        ]
    )
    dq_run_kpis_path = OUT_DIR / "dq_run_kpis.csv"
    dq_run_kpis.to_csv(dq_run_kpis_path, index=False)
    out_paths["dq_run_kpis_csv"] = str(dq_run_kpis_path)

    metadata = {
        "pipeline_version": PIPELINE_VERSION,
        "run_id": run_id,
        "run_utc": utc_now_iso(),
        "input": {
            "path": str(RAW_CSV_PATH),
            "sha256": input_hash,
            "row_count_input_extracted": rows_extracted_input,
            "row_count_after_dedup": rows_after_dedup,
            "rows_with_valid_id_post_dedup": post_dedup_rows_with_id,
            "post_dedup_unique_ids": post_dedup_unique_ids,
            "complaint_id_unique_ok_for_valid_ids": complaint_id_unique_ok,
            "csv_read_audit": read_meta,
        },
        "date_parsing": date_parsing_meta,
        "windowing": window_meta,
        "dedupe": dedupe_meta,
        "local_validation": {
            "years_back_anchor": YEARS_BACK,
            "allowed_domains": sorted(list(ALLOWED_DOMAINS)),
            "scope_require_narrative": SCOPE_REQUIRE_NARRATIVE,
            "scope_diagnostics_rates": scope_diagnostics_rates,
            "narrative_present_rate_scoped": narrative_present_rate_scoped,
        },
        "data_quality_core": {
            "missing_complaint_id_count": missing_complaint_id_count,
            "missing_lag_input_rate": missing_lag_input_rate,
            "negative_lag_count_in_window": negative_lag_count_in_window,
            "negative_lag_count_scoped": negative_lag_count_scoped,
            "negative_lag_rate_scoped": negative_lag_rate_scoped,
            "narrative_p75_chars_scoped": narrative_p75_chars,
            "blank_id_count": blank_id_count,
            "duplicates_raw_count_valid_ids": duplicates_raw_count,
            "unmapped_focus_candidate_rate_in_window": unmapped_focus_rate,
            "unmapped_focus_candidate_count": unmapped_focus_count,
            "focus_candidate_count": focus_candidate_count,
            "filter_impact_counts": {
                "removed_not_in_window": removed_not_in_window,
                "removed_out_of_scope_domain_in_window": removed_out_of_scope_domain_in_window,
                "diagnostic_missing_narrative_in_window": diagnostic_missing_narrative_in_window,
                "scope_removed_missing_narrative": scope_removed_missing_narrative,
            },
        },
        "outputs": {
            "out_dir": str(OUT_DIR),
            "primary_database_import_artifact": "postgres_csv",
            "postgres_csv_encoding": POSTGRES_CSV_ENCODING,
            "postgres_csv_header": True,
            "postgres_csv_null_representation": "empty_field",
            "postgres_files": {
                "stage": POSTGRES_STAGE_FILENAME if EXPORT_POSTGRES_CSV else None,
                "tag_bridge": POSTGRES_TAG_FILENAME if EXPORT_POSTGRES_CSV else None,
            },
            "parquet_sor_enabled": EXPORT_PARQUET_SOR,
            "files_written": out_paths,
            "row_counts": {
                "stg_scoped_rows": rows_scoped,
                "bridge_scoped_rows": int(len(bridge_scoped)),
            },
        },
        "notes": [
            "PostgreSQL CSV exports are the primary database import artifacts.",
            "Date columns in PostgreSQL stage CSV are exported as YYYY-MM-DD strings.",
            "Missing values in CSV are written as empty fields for PostgreSQL COPY compatibility.",
            "Parquet remains optional as system-of-record archival output.",
            "Legacy SQL Server bulk-compatible export is optional and disabled by default.",
            "CSV read audit captures missing requested columns and bad-line policy.",
            "Dedupe uses best-record tie-break on duplicated complaint_id only.",
            "Scoped stage builder is shared with date-mode switch for parquet vs CSV exports.",
        ],
    }
    write_json(OUT_DIR / "pipeline_run_metadata.json", metadata)

    logger.info("Wrote outputs:")
    for k, p in out_paths.items():
        logger.info("- %s: %s", k, p)
    logger.info("Wrote: %s", OUT_DIR / "pipeline_run_metadata.json")


if __name__ == "__main__":
    main()

2026-04-23 19:41:09,175 | INFO | Reading input CSV...
2026-04-23 19:45:29,753 | INFO | Rows extracted: 12691672
2026-04-23 19:45:40,670 | INFO | Running dedupe...
2026-04-23 20:03:35,520 | INFO | Wrote outputs:
2026-04-23 20:03:35,530 | INFO | - stg_scoped_parquet: C:\DAB\sem2\env\project\Clean\stg_complaints_scoped.parquet
2026-04-23 20:03:35,530 | INFO | - bridge_scoped_parquet: C:\DAB\sem2\env\project\Clean\bridge_complaint_tag_scoped.parquet
2026-04-23 20:03:35,531 | INFO | - stg_scoped_postgres_csv: C:\DAB\sem2\env\project\Clean\stg_complaints_scoped_postgres.csv
2026-04-23 20:03:35,531 | INFO | - bridge_scoped_postgres_csv: C:\DAB\sem2\env\project\Clean\bridge_complaint_tag_scoped_postgres.csv
2026-04-23 20:03:35,532 | INFO | - sample_scoped_csv: C:\DAB\sem2\env\project\Clean\sample_scoped_100k.csv
2026-04-23 20:03:35,532 | INFO | - dq_run_kpis_csv: C:\DAB\sem2\env\project\Clean\dq_run_kpis.csv
2026-04-23 20:03:35,534 | INFO | Wrote: C:\DAB\sem2\env\project\Clean\pipeline_run_met